In [1]:
import pandas as pd
import numpy as np
data={
     'farm_id':['f001','f002','f003','f004','f005','f006','f007'],
     'crop':['wheat','rice','corn','wheat','rice','corn','wheat'],
     'yield_tons':[180,228,416,164,245,390,195],
     'rainfall_mm':[450,820,680,420,850,710,480],
     'fertilizer_kg':[120,150,200,np.nan,160,195,125],
     'soil_ph':[6.5,5.8,6.2,6.7,5.9,6.2,6.4]
     }

df=pd.DataFrame(data)
print('original data with missing value:')
print(df)

print('\n ===Missing value analysis===')
print('\n missing value per column:')
print(df.isna().sum())
print(f'\n Total missing values:{df.isna().sum().sum()}')
print(f'percentage of missing data:{(df.isna().sum().sum()/df.size*100):.2f}%')

print('\n=== ROWS WITH THE MISSING DATA==')
rows_with_missing=df[df.isna().any(axis=1)]
print(rows_with_missing)

#strategy1:drop rows with missing values
print('\n===strategy 1:drop missing rows')
df_dropped=df.dropna()
print(f'original shape:{df.shape}')
print(f'after dropping:{df_dropped.shape}')
print(df_dropped)

#strategy 2:fill with mean/median
print('\n ===strategy 2:fill with the statistics')
df_filled=df.copy()
#fill nummeric columns with means
df_filled['yield_tons'].fillna(df_filled['yield_tons'].mean(),inplace=True)
df_filled['rainfall_mm'].fillna(df_filled['rainfall_mm'].mean(),inplace=True)
df_filled['fertilizer_kg'].fillna(df_filled['fertilizer_kg'].mean(),inplace=True)
df_filled['soil_ph'].fillna(df_filled['soil_ph'].mean(),inplace=True)
#fill categorical values with mode
df_filled['crop'].fillna(df_filled['crop'].mode()[0],inplace=True)
print(df_filled)

#strategy 3:forward fill(useful for time series)
print('\n===strategy 3:forward fill')
df_ffill=df.fillna(method='ffill')
print(df_ffill)

#strategy 4:custom fill per column
print('\n===strtegy 4:custom fill===')
df_custom=df.fillna({
    'crop':'unknown',
    'yield_tons':df['yield_tons'].mean(),
    'rainfall_mm':650,#regional averege
    'fertilizer_kg':150,
    'soil_ph':6.3
    })
print(df_custom)

print('\n===verification===')
print(f'missing values after treatment:{df_custom.isna().sum().sum()}')

original data with missing value:
  farm_id   cpro  yield_tons  rainfall_mm  fertilizer_kg  soil_ph
0    f001  wheat         180          450          120.0      6.5
1    f002   rice         228          820          150.0      5.8
2    f003   corn         416          680          200.0      6.2
3    f004  wheat         164          420            NaN      6.7
4    f005   rice         245          850          160.0      5.9
5    f006   corn         390          710          195.0      6.2
6    f007  wheat         195          480          125.0      6.4

 ===Missing value analysis===

 missing value per column:
farm_id          0
cpro             0
yield_tons       0
rainfall_mm      0
fertilizer_kg    1
soil_ph          0
dtype: int64

 Total missing values:1
percentage of missing data:2.38%

=== ROWS WITH THE MISSING DATA==
  farm_id   cpro  yield_tons  rainfall_mm  fertilizer_kg  soil_ph
3    f004  wheat         164          420            NaN      6.7

===strategy 1:drop missing 

/tmp/ipython-input-3428705273.py:37: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_filled['yield_tons'].fillna(df_filled['yield_tons'].mean(),inplace=True)
/tmp/ipython-input-3428705273.py:38: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(

In [ ]:
#DATA AGGREGATION AND GROUPING
import pandas as pd
#multi-year farm data
data={
    'year':[2020,2020,2020,2021,2021,2021,2022,2022,2022,2023,2023,2023],
    'region':['north','south','east','north','south','east','north','south','east','north','south','east'],
    'crop':['wheat','rice','corn','wheat','rice','corn','wheat','rice','corn','wheat','rice','corn'],
    'yield_tons':[180,228,416,195,245,430,210,250,445,225,260,455],
    'area_ha':[50,45,55,52,46,56,54,48,57,55,49,58],
    'cost_thousand':[85,95,180,90,100,190,95,105,195,100,110,200]
    }
df=pd.DataFrame(data)
print('==agricultural data set==')
print(df)

#group by region
print('\n===analysis by region===')
region_stats=df.groupby('region').agg({
    'yield_tons':'sum',
    'area_ha':'sum',
    'cost_thousand':'sum'
    })
print(region_stats)

#group by crop
print('\n=== analysis by crop===')
crop_stats=df.groupby('crop').agg({
    'yield_tons':['mean','max','min'],
    'cost_thousand':'mean'
    }).round(2)
print(crop_stats)

#multiple grouping region
print('\n===region crop analysis===')
region_crop=df.groupby(['region','crop'])['yield_tons'].mean().round(2)
print(region_crop)

#year over growth
print('\n ===yearly trends===')
yearly=df.groupby('year').agg({
    'yield_tons':'sum',
    'area_ha':'sum'
    })
yearly['yield_per_ha']=(yearly['yield_tons']/yearly['area_ha']).round(2)
print(yearly)

#transform :add group mean to each row
df['region_avg_yield']=df.groupby('region')['yield_tons'].transform('mean').round(2)
df['crop_avg_yield']=df.groupby('crop')['yield_tons'].transform('mean').round(2)
print('\n===transformed data===')
print(df)

print('\n === DATASET WITH GROUP AVERAGES===')
print(df[['year','region','crop','yield_tons','region_avg_yield','crop_avg_yield']].head(6))

#pivot table
print('\n===pivot table:average yield by region and crop===')
pivot=df.pivot_table(values='yield_tons',index='region',columns='crop',aggfunc='mean').round(2)
print(pivot)

#filter groups with high performance
print('\n===high performing regions(avg yield >300)===')
high_regions=df.groupby('region').filter(lambda x:x['yield_tons'].mean()>300)
print(high_regions[['region','crop','yield_tons']].drop_duplicates('region'))

==agricultural data set==
    year region   crop  yield_tons  area_ha  cost_thousand
0   2020  north  wheat         180       50             85
1   2020  south   rice         228       45             95
2   2020   east   corn         416       55            180
3   2021  north  wheat         195       52             90
4   2021  south   rice         245       46            100
5   2021   east   corn         430       56            190
6   2022  north  wheat         210       54             95
7   2022  south   rice         250       48            105
8   2022   east   corn         445       57            195
9   2023  north  wheat         225       55            100
10  2023  south   rice         260       49            110
11  2023   east   corn         455       58            200

===analysis by region===
        yield_tons  area_ha  cost_thousand
region                                    
east          1746      226            765
north          810      211            370
south    